In [ ]:
import numpy as np
import keras
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from keras.datasets import imdb
from time import time

from tmu.models.classification.vanilla_classifier import TMClassifier

hypervector_size = 10000

NUM_WORDS=5000
INDEX_FROM=2 

print("Downloading dataset...")

train,test = keras.datasets.imdb.load_data(num_words=NUM_WORDS, index_from=INDEX_FROM)

train_x,train_y = train
test_x,test_y = test

word_to_id = keras.datasets.imdb.get_word_index()
word_to_id = {k:(v+INDEX_FROM) for k,v in word_to_id.items()}
word_to_id["<PAD>"] = 0
word_to_id["<START>"] = 1
word_to_id["<UNK>"] = 2

print("Producing bit representation...")

encoding = np.random.choice([-1,1], size=(NUM_WORDS, hypervector_size)).astype(np.int32)

X_train = np.zeros((train_y.shape[0], hypervector_size), dtype=np.int32)
Y_train = np.zeros(train_y.shape[0], dtype=np.uint32)
for i in range(train_y.shape[0]):
	seen = {}
	for word_id in train_x[i]:
		if word_id not in seen:
			X_train[i] += encoding[word_id]
			seen[word_id] = True
	Y_train[i] = train_y[i]
X_train = np.where(X_train >= 0, 1, 0).astype(np.uint32)

X_test = np.zeros((test_y.shape[0], hypervector_size), dtype=np.int32)
Y_test = np.zeros(test_y.shape[0], dtype=np.uint32)
for i in range(test_y.shape[0]):
	seen = {}
	for word_id in test_x[i]:
		if word_id not in seen:
			X_test[i] += encoding[word_id]
			seen[word_id] = True
	Y_test[i] = test_y[i]
X_test = np.where(X_test >= 0, 1, 0).astype(np.uint32)

tm = TMClassifier(10000, 8000, 5.0, platform='CUDA', weighted_clauses=True, clause_drop_p=0.75)

print("\nAccuracy over 40 epochs:\n")
for i in range(40):
	start_training = time()
	tm.fit(X_train, Y_train)
	stop_training = time()

	start_testing = time()
	result = 100*(tm.predict(X_test) == Y_test).mean()
	stop_testing = time()

	print("#%d Accuracy: %.2f%% Training: %.2fs Testing: %.2fs" % (i+1, result, stop_training-start_training, stop_testing-start_testing))
